In [1]:
import sys
sys.path.append("../src")
# or pip install -e. in root (cleaner)

In [2]:
from anomaly_detection.data.data import DataModule  

In [5]:
from anomaly_detection.data.data import DataModule
    
from anomaly_detection.infra.selection.candidate_registry import CandidateRegistry
from anomaly_detection.infra.selection.model_selector import ModelSelector
from pathlib import Path

from anomaly_detection.inference.benchmarking import benchmark_candidates

## candidae models

In [6]:
# Load data
#data = DataModule(TRAIN_PATH, VAL_PATH, Y_VAL_PATH)
#X_train, X_val, y_val = data.load()


import pandas as pd

# Registry
from pathlib import Path

root_dir = Path.cwd().parents[0]
tracking_db = root_dir / "mlflow.db"
candidate_db_url = f"sqlite:///{tracking_db}"

registry = CandidateRegistry(candidate_db_url)

# Candidate table;M later a get_candiudates will be better
candidates = registry.print_candidates(
    experiment_id=1,
    include_evicted=True,
)

candidates_df = pd.DataFrame(candidates)

display(candidates_df)

# Benchmark
"""
benchmark_candidates(
    registry=registry,
    experiment_id=1,
    X_benchmark=X_val[:100],
)
"""

# Selection
selector = ModelSelector(registry)

selected = selector.select(
    experiment_id=1,
    pr_auc_tolerance=0.005,
)

print("Selected candidate:")
print(selected)


Experiment: 1

Rank  Model          PR-AUC    State       Run ID
----------------------------------------------------------------------
1     isoforest      0.9982    retained    c16d41cfcb084fc6a24839f63e4a8341
2     isoforest      0.9967    retained    09046008605244eeb0144356dbce9aac
3     transformer    0.9690    retained    8c2d0d0139a042dd9752e5360dbb4580
4     vae            0.9679    retained    f5fa3ccc979748d782a3838ea65d72a6
5     ae             0.9658    retained    17e3053a24df450388d580d77273b502
6     ae             0.9622    evicted     eac5116ba441458c8789cd2c43f6dd82
7     transformer    0.9192    evicted     797581862eb44372976b95fa3bd4e765



""


Selected candidate:


In [7]:
import pandas as pd

records = registry.candidate_records(
    experiment_id=1,
    include_evicted=True,
)

candidates_df = pd.DataFrame(records)

candidates_df.insert(
    0,
    "rank",
    range(1, len(candidates_df) + 1),
)

display(candidates_df)

,rank,run_id,model_family,val_pr_auc,inference_ms,explainability,state,artifact_path,created_at
0,1,c16d41cfcb084fc6a24839f63e4a8341,isoforest,0.998188,18.530554,None,retained,model,2026-09-09 17:53:25.393240
1,2,09046008605244eeb0144356dbce9aac,isoforest,0.996682,18.060460,None,retained,model,2026-09-09 17:53:24.729601
2,3,8c2d0d0139a042dd9752e5360dbb4580,transformer,0.968982,5.027218,None,retained,model,2026-09-09 17:53:35.098863
3,4,f5fa3ccc979748d782a3838ea65d72a6,vae,0.967894,1.112328,None,retained,model,2026-09-09 17:53:01.634873
4,5,17e3053a24df450388d580d77273b502,ae,0.965830,1.463905,None,retained,model,2026-09-11 19:23:47.634436
5,6,eac5116ba441458c8789cd2c43f6dd82,ae,0.962170,1.566400,None,evicted,model,2026-09-09 17:51:41.758027
6,7,797581862eb44372976b95fa3bd4e765,transformer,0.919243,NaN,None,evicted,model,2026-09-09 17:53:05.180203


## test preds (and later merics with my bets model)

In [21]:


# preds!!!!

import numpy as np

from pathlib import Path

from anomaly_detection.inference.loader import load_from_config

import json


def main():


    with open("/home/marcos/Escritorio/AI-prod/Anomaly-Detection/mock_model_store/models/model_001/config.json", "r") as f:
        config = json.load(f)


    #model_dir = Path("mock_model_store/models/model_001")
    model_dir = Path("/home/marcos/Escritorio/AI-prod/Anomaly-Detection/mock_model_store/models/model_001")

    runner = load_from_config(
        config=config,
        model_dir=model_dir,
    )


    X = np.random.randn(20, 11)

    predictions = runner.predict(X)

    print("Input shape:", X.shape)
    print("Predictions:")
    print(predictions)

In [22]:
main()

Input shape: (20, 11)
Predictions:
[0 0 0 0 0 0 0 0 0 0 0]
